# Detecting Anomalous Energy Performance in UK Buildings
### Identifying Inefficient Properties for Retrofit Prioritisation

**Author:** Yenlik Gaisina, MPH | Cambridge Data Science & AI
**Data:** UK Government EPC Open Data  --  Ministry of Housing, Communities & Local Government
**Methods:** IQR  |  One-Class SVM  |  Isolation Forest  |  PCA

---
## Table of Contents
1. [Data Import & Cleaning](#data-import)
2. [Initial EDA](#initial-eda)
3. [Visualisations](#visualisations)
4. [IQR-Based Anomaly Detection](#iqr-flags)
5. [One-Class SVM](#one-class-svm)
6. [Isolation Forest](#isolation-forest)
7. [Method Agreement Analysis](#method-agreement)
8. [Anomaly Profiling & Retrofit Prioritisation](#anomaly-profiling)
9. [Comparison & Recommendation](#comparison)

## 1. Data Import & Cleaning

Real domestic EPC data downloaded from the UK Government's [EPC Open Data](https://epc.opendatacommunities.org/) API.
Register (free) at the link above to obtain an API key; the key is emailed immediately.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.preprocessing import StandardScaler
from sklearn.svm import OneClassSVM
from sklearn.ensemble import IsolationForest
from sklearn.decomposition import PCA

import urllib.request, base64, io, os

# ── EPC API credentials ────────────────────────────────────────────────────
# Register (free) at https://epc.opendatacommunities.org/ to get your key.
EPC_EMAIL   = 'your-email@example.com'   # <-- replace with your email
EPC_API_KEY = 'your-api-key-here'        # <-- replace with your API key

CACHE_FILE  = 'epc_domestic_sample.csv'
BASE_URL    = 'https://epc.opendatacommunities.org/api/v1/domestic/search'

# Four geographically diverse local authorities (~5,000 records each)
LOCAL_AUTHORITIES = {
    'E08000025': 'Birmingham',
    'E08000035': 'Leeds',
    'E06000018': 'Nottingham',
    'E09000012': 'Hackney',
}

auth_token = base64.b64encode(f'{EPC_EMAIL}:{EPC_API_KEY}'.encode()).decode()
headers = {
    'Authorization': f'Basic {auth_token}',
    'Accept': 'text/csv',
}

if os.path.exists(CACHE_FILE):
    print(f'Loading cached data from {CACHE_FILE}')
    df_raw = pd.read_csv(CACHE_FILE, low_memory=False)
else:
    print('Downloading domestic EPC certificates from four local authorities...')
    all_frames = []
    for la_code, la_name in LOCAL_AUTHORITIES.items():
        url = f'{BASE_URL}?size=5000&local-authority={la_code}'
        req = urllib.request.Request(url, headers=headers)
        with urllib.request.urlopen(req, timeout=120) as resp:
            chunk = pd.read_csv(io.StringIO(resp.read().decode('utf-8')))
        all_frames.append(chunk)
        print(f'  {la_name} ({la_code}): {len(chunk):,} certificates')

    df_raw = pd.concat(all_frames, ignore_index=True)

    # Drop personal data columns (GDPR compliance)
    personal = [c for c in df_raw.columns
                if any(x in c.lower() for x in ['address', 'postcode', 'uprn', 'lmk-key'])]
    df_raw.drop(columns=personal, inplace=True, errors='ignore')

    # Rename: hyphens -> underscores (Pandas-friendly)
    df_raw.columns = [c.replace('-', '_') for c in df_raw.columns]

    df_raw.to_csv(CACHE_FILE, index=False)
    print(f'\nSaved {len(df_raw):,} records to {CACHE_FILE} (personal data removed)')

print(f'\nRaw dataset: {df_raw.shape[0]:,} rows x {df_raw.shape[1]} columns')
df_raw.head(3)

In [ ]:
# ── Select features for anomaly detection ──────────────────────────────────
epc_cols = [
    'current_energy_efficiency',    # SAP rating (1-100)
    'energy_consumption_current',   # kWh/m2/yr
    'co2_emissions_current',        # tonnes CO2/yr
    'total_floor_area',             # m2
    'heating_cost_current',         # GBP/yr
    'hot_water_cost_current',       # GBP/yr
    'lighting_cost_current',        # GBP/yr
]

profile_cols = [
    'construction_age_band',
    'property_type',
    'current_energy_rating',
    'walls_description',
    'main_fuel',
]

keep_cols = epc_cols + profile_cols
df = df_raw[[c for c in keep_cols if c in df_raw.columns]].copy()

# Clean construction_age_band: remove 'England and Wales: ' prefix
if 'construction_age_band' in df.columns:
    df['construction_age_band'] = (df['construction_age_band']
                                   .str.replace('England and Wales: ', '', regex=False)
                                   .str.strip())

# Drop rows with missing values in numeric features
before = len(df)
df = df.dropna(subset=epc_cols)
after = len(df)
print(f'Rows before cleaning: {before:,}')
print(f'Rows after dropping NaN in numeric features: {after:,}')
print(f'Dropped: {before - after:,} ({100*(before-after)/before:.1f}%)')

# Convert numeric columns
for c in epc_cols:
    df[c] = pd.to_numeric(df[c], errors='coerce')

# Remove physically impossible values (negative costs, zero floor area)
df = df[(df['total_floor_area'] > 0) & (df['heating_cost_current'] >= 0)]

print(f'Final dataset: {len(df):,} domestic EPC certificates')
df.head()

## 2. Initial EDA
Missing values, duplicates, descriptive statistics

In [ ]:
df[epc_cols].info()
print()
print('=== Missing Values (numeric features) ===')
print(df[epc_cols].isna().sum())
print()
print('=== Duplicate Rows ===')
print(f'{df.duplicated().sum():,} duplicates')

**EDA Observations.**
After cleaning, the dataset contains real domestic EPC certificates from four English local authorities (Birmingham, Leeds, Nottingham, Hackney). All seven numeric features are complete after the cleaning step. Any remaining duplicates are genuine re-assessments of the same property at different dates, which is expected in EPC data.

## 3. Visualisations
Descriptive statistics, distributions, correlations, and construction era analysis

In [ ]:
desc = df[epc_cols].describe().T
desc['median'] = df[epc_cols].median()
desc[['count','mean','median','std','min','25%','75%','max']]

### Descriptive statistics observations

The descriptive statistics reveal the typical ranges for domestic energy performance in England. Key observations:

- **Energy efficiency** (SAP score 1-100): the mean and median indicate the typical domestic property sits in the D-E band range, consistent with the national housing stock profile.
- **Energy consumption** and **CO2 emissions** both show substantial right-hand skew, with maximum values several times the mean -- indicating the presence of high-consumption outliers.
- **Heating cost** dominates total energy expenditure, typically 5-8x larger than lighting or hot water costs.
- **Total floor area** varies widely, from small flats to large detached properties, which directly influences energy demand.

In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(18, 8))
axes = axes.flatten()

for i, col in enumerate(epc_cols):
    axes[i].hist(df[col], bins=60, edgecolor='white', linewidth=0.4, color='steelblue')
    label = col.replace('_', ' ').title()
    axes[i].set_title(label, fontsize=9, fontweight='bold')
    axes[i].set_xlabel('Value')
    axes[i].set_ylabel('Frequency')

axes[-1].axis('off')   # hide unused 8th subplot
plt.suptitle('EPC Feature Distributions', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

fig2, axes2 = plt.subplots(2, 4, figsize=(18, 8))
axes2 = axes2.flatten()

for i, col in enumerate(epc_cols):
    bp = axes2[i].boxplot(df[col], vert=True, patch_artist=True,
                          boxprops=dict(facecolor='steelblue', alpha=0.6))
    label = col.replace('_', ' ').title()
    axes2[i].set_title(label, fontsize=9, fontweight='bold')
    axes2[i].set_ylabel('Value')

axes2[-1].axis('off')
plt.suptitle('EPC Feature Boxplots  --  Outliers Visible Beyond Whiskers', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

### Distribution observations

The histograms confirm that most EPC features follow approximately unimodal distributions. Several features exhibit long right-hand tails, particularly `energy_consumption_current`, `heating_cost_current`, and `co2_emissions_current`, indicating properties with extreme values.

The boxplots clearly highlight outliers beyond the upper quartile for multiple features. These extreme values suggest properties with significantly higher energy consumption, costs, or emissions than typical -- and are the primary targets for anomaly detection.

In [ ]:
# ── Correlation heatmap ──────────────────────────────────────────────────
corr = df[epc_cols].corr()
mask = np.triu(np.ones_like(corr, dtype=bool))

fig, ax = plt.subplots(figsize=(9, 7))
sns.heatmap(corr, mask=mask, annot=True, fmt='.2f', cmap='coolwarm',
            center=0, linewidths=0.5, ax=ax,
            xticklabels=[c.replace('_', ' ').title() for c in epc_cols],
            yticklabels=[c.replace('_', ' ').title() for c in epc_cols])
ax.set_title('Feature Correlation Matrix', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

### Correlation observations

Energy consumption, CO2 emissions, and heating cost are strongly positively correlated, as expected -- buildings that consume more energy produce more CO2 and cost more to heat. Total floor area shows moderate positive correlation with all cost features (larger buildings cost more). Current energy efficiency (SAP score) is negatively correlated with consumption and emissions -- higher-rated buildings are more efficient. These correlations confirm the features capture overlapping but complementary aspects of building energy performance, supporting a multivariate anomaly detection approach.

In [ ]:
# ── Construction era distribution ────────────────────────────────────────
if 'construction_age_band' in df.columns:
    era_order = ['before 1900', '1900-1929', '1930-1949', '1950-1966',
                 '1967-1975', '1976-1982', '1983-1990', '1991-1995',
                 '1996-2002', '2003-2006', '2007-2011', '2012 onwards']
    era_counts = df['construction_age_band'].value_counts()
    # Reindex to chronological order (only include eras present in data)
    era_order_present = [e for e in era_order if e in era_counts.index]
    era_counts = era_counts.reindex(era_order_present).dropna()

    fig, axes = plt.subplots(1, 2, figsize=(16, 6))

    # Left: count by era
    axes[0].barh(era_counts.index, era_counts.values, color='steelblue', alpha=0.85)
    axes[0].set_xlabel('Number of certificates')
    axes[0].set_title('EPC Certificates by Construction Era', fontweight='bold')
    axes[0].invert_yaxis()

    # Right: median energy consumption by era
    era_consumption = df.groupby('construction_age_band')['energy_consumption_current'].median()
    era_consumption = era_consumption.reindex(era_order_present).dropna()
    colors = ['#c0392b' if v > era_consumption.median() else 'steelblue'
              for v in era_consumption.values]
    axes[1].barh(era_consumption.index, era_consumption.values, color=colors, alpha=0.85)
    axes[1].set_xlabel('Median energy consumption (kWh/m2/yr)')
    axes[1].set_title('Median Energy Consumption by Construction Era\n(red = above median)',
                      fontweight='bold')
    axes[1].invert_yaxis()

    plt.tight_layout()
    plt.show()

### Construction era observations

Pre-1930 properties show the highest median energy consumption, consistent with solid-wall construction, poor insulation, and single-glazed windows. Properties built after 1982 show progressively lower energy consumption, reflecting successive tightening of Building Regulations (Part L). The sharp improvement after 2006 corresponds to the introduction of the Code for Sustainable Homes. This confirms that construction era is a strong predictor of energy inefficiency and should correlate with anomaly detection results.

## 4. IQR-Based Anomaly Detection

In [ ]:
def iqr_bounds(s):
    q1 = s.quantile(0.25)
    q3 = s.quantile(0.75)
    iqr = q3 - q1
    return q1 - 1.5 * iqr, q3 + 1.5 * iqr

flag_cols = []
for c in epc_cols:
    lower, upper = iqr_bounds(df[c])
    fc = f'{c}_flag'
    df[fc] = ((df[c] < lower) | (df[c] > upper)).astype(int)
    flag_cols.append(fc)
    flagged_pct = df[fc].mean() * 100
    print(f'{c:35s}  bounds=[{lower:8.1f}, {upper:8.1f}]  flagged={flagged_pct:.1f}%')

In [ ]:
df['n_flags'] = df[flag_cols].sum(axis=1)
n_rows = len(df)
summary = []
for k in range(1, len(epc_cols) + 1):
    count = (df['n_flags'] >= k).sum()
    pct = 100 * count / n_rows
    summary.append((k, count, round(pct, 2)))

iqr_summary = pd.DataFrame(summary, columns=['k_or_more_flags', 'n_rows', 'pct_rows'])
iqr_summary

k=1 flags too high a proportion (above 5%). k=2 lands within 1-5%, so properties with two or more simultaneously flagged features are treated as anomalous. This reflects the principle that a single unusual reading may be noise, but multiple abnormal features occurring together are more indicative of genuine inefficiency.

In [ ]:
k_chosen = 2
df['iqr_anomaly'] = (df['n_flags'] >= k_chosen).astype(int)
iqr_pct = df['iqr_anomaly'].mean() * 100
print(f'IQR anomalies (k >= {k_chosen}): {df["iqr_anomaly"].sum():,} properties ({iqr_pct:.1f}%)')

### IQR method observations

Requiring at least two features to simultaneously be in an outlier condition (k=2) reduces the anomaly proportion to the expected 1-5% range. This threshold reflects the business context: in building energy analysis, combinations of abnormal readings -- for example, very high energy consumption alongside high CO2 emissions -- are more meaningful indicators of genuine inefficiency than any single metric in isolation.

## 5. One-Class SVM
Scaling, parameter tuning, PCA visualisation

In [ ]:
X = df[epc_cols].copy()

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

def ocsvm_outlier_pct(nu, gamma):
    model = OneClassSVM(kernel='rbf', nu=nu, gamma=gamma)
    y_pred = model.fit_predict(X_scaled)
    return (y_pred == -1).mean() * 100, y_pred

nu_list = [0.01, 0.02, 0.03, 0.05]
gamma_list = ['scale', 0.1, 0.01]

results = []
saved = {}

for nu in nu_list:
    for gamma in gamma_list:
        pct, y_pred = ocsvm_outlier_pct(nu, gamma)
        results.append((nu, gamma, round(pct, 2)))
        saved[(nu, gamma)] = y_pred

svm_grid = pd.DataFrame(results, columns=['nu', 'gamma', 'pct_outliers'])
svm_grid.sort_values('pct_outliers')

In [ ]:
# Select configuration closest to ~2% outlier rate
best_nu = 0.02
best_gamma = 'scale'
y_pred_svm = saved[(best_nu, best_gamma)]
df['svm_anomaly'] = (y_pred_svm == -1).astype(int)

svm_pct = df['svm_anomaly'].mean() * 100
print(f'One-Class SVM anomalies (nu={best_nu}, gamma={best_gamma}): {df["svm_anomaly"].sum():,} ({svm_pct:.1f}%)')

### One-Class SVM observations

The selected configuration (`nu=0.02`, `gamma='scale'`) identifies approximately 2% of properties as anomalous, falling within the expected 1-5% range. The SVM models a decision boundary in the standardised feature space, capturing non-linear relationships between energy features that the IQR method cannot detect.

### PCA Visualisation

In [ ]:
pca = PCA(n_components=2, random_state=42)
X_pca = pca.fit_transform(X_scaled)
var_explained = pca.explained_variance_ratio_.sum() * 100

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for ax, (labels, title, cmap_name) in zip(axes, [
    (df['svm_anomaly'].values, 'One-Class SVM', 'coolwarm'),
    (df['iqr_anomaly'].values, 'IQR (k>=2)', 'coolwarm'),
]):
    scatter = ax.scatter(X_pca[:, 0], X_pca[:, 1], c=labels,
                         cmap=cmap_name, s=5, alpha=0.4)
    ax.set_xlabel(f'PC1 ({pca.explained_variance_ratio_[0]*100:.1f}%)')
    ax.set_ylabel(f'PC2 ({pca.explained_variance_ratio_[1]*100:.1f}%)')
    ax.set_title(f'PCA (2D)  --  {title} Anomalies', fontweight='bold')

plt.suptitle(f'Total variance explained by 2 components: {var_explained:.1f}%',
             fontsize=11, y=1.02)
plt.tight_layout()
plt.show()

### PCA observations

The first two principal components explain a moderate share of total variance. Anomalous properties tend to cluster in the high-PC1 region, driven primarily by correlated energy consumption, CO2 emissions, and heating cost features. However, the separation is not absolute in two dimensions -- the anomalous points overlap with the normal cluster periphery, indicating that the relevant variation is distributed across more than two components. PCA is therefore used as a supporting exploratory tool, not a definitive validation method.

## 6. Isolation Forest
Parameter tuning and PCA visualisation

In [ ]:
def iforest_outlier_pct(contam):
    model = IsolationForest(contamination=contam, n_estimators=200, random_state=42)
    y_pred = model.fit_predict(X_scaled)
    return (y_pred == -1).mean() * 100, y_pred

contams = [0.01, 0.02, 0.03, 0.05]
if_results = []
if_saved = {}

for c in contams:
    pct, y_pred = iforest_outlier_pct(c)
    if_results.append((c, round(pct, 2)))
    if_saved[c] = y_pred

if_grid = pd.DataFrame(if_results, columns=['contamination', 'pct_outliers'])
if_grid

In [ ]:
best_contam = 0.02
y_pred_if = if_saved[best_contam]
df['if_anomaly'] = (y_pred_if == -1).astype(int)

if_pct = df['if_anomaly'].mean() * 100
print(f'Isolation Forest anomalies (contamination={best_contam}): {df["if_anomaly"].sum():,} ({if_pct:.1f}%)')

# PCA plot
fig, ax = plt.subplots(figsize=(8, 5))
scatter = ax.scatter(X_pca[:, 0], X_pca[:, 1], c=df['if_anomaly'].values,
                     cmap='coolwarm', s=5, alpha=0.4)
ax.set_xlabel(f'PC1 ({pca.explained_variance_ratio_[0]*100:.1f}%)')
ax.set_ylabel(f'PC2 ({pca.explained_variance_ratio_[1]*100:.1f}%)')
ax.set_title('PCA (2D)  --  Isolation Forest Anomalies', fontweight='bold')
plt.tight_layout()
plt.show()

### Isolation Forest observations

A contamination value of 0.02 identifies approximately 2% of properties as anomalous, consistent with the IQR and SVM methods. Isolation Forest achieves this with minimal tuning -- the contamination parameter directly specifies the expected anomaly proportion, making it the easiest of the three methods to calibrate. The PCA plot shows a similar pattern to the SVM result, with anomalous properties concentrated in the high-consumption region of PC1.

## 7. Method Agreement Analysis
Cross-validating anomaly detection without labelled ground truth

In [ ]:
# ── Method agreement matrix ──────────────────────────────────────────────
methods = {'IQR (k>=2)': 'iqr_anomaly', 'One-Class SVM': 'svm_anomaly', 'Isolation Forest': 'if_anomaly'}

print('=== Anomaly counts by method ===')
for name, col in methods.items():
    n = df[col].sum()
    print(f'  {name:20s}: {n:,} ({100*n/len(df):.1f}%)')

# Pairwise overlap
print('\n=== Pairwise agreement (% of union flagged by both) ===')
method_names = list(methods.keys())
method_cols = list(methods.values())
for i in range(len(method_names)):
    for j in range(i+1, len(method_names)):
        both = ((df[method_cols[i]] == 1) & (df[method_cols[j]] == 1)).sum()
        either = ((df[method_cols[i]] == 1) | (df[method_cols[j]] == 1)).sum()
        overlap = 100 * both / either if either > 0 else 0
        print(f'  {method_names[i]:20s} x {method_names[j]:20s}: {both:,} shared ({overlap:.1f}% Jaccard)')

# High-confidence tier: flagged by all three methods
df['all_three'] = ((df['iqr_anomaly'] == 1) & (df['svm_anomaly'] == 1) & (df['if_anomaly'] == 1)).astype(int)
df['any_two'] = ((df['iqr_anomaly'] + df['svm_anomaly'] + df['if_anomaly']) >= 2).astype(int)

print(f'\nHigh-confidence (all 3 methods): {df["all_three"].sum():,} properties')
print(f'Medium-confidence (any 2 methods): {df["any_two"].sum():,} properties')

# Venn-style bar chart
fig, ax = plt.subplots(figsize=(10, 5))
agree_labels = ['IQR only', 'SVM only', 'IF only', 'Any 2 agree', 'All 3 agree']
agree_counts = [
    ((df['iqr_anomaly']==1) & (df['svm_anomaly']==0) & (df['if_anomaly']==0)).sum(),
    ((df['iqr_anomaly']==0) & (df['svm_anomaly']==1) & (df['if_anomaly']==0)).sum(),
    ((df['iqr_anomaly']==0) & (df['svm_anomaly']==0) & (df['if_anomaly']==1)).sum(),
    df['any_two'].sum() - df['all_three'].sum(),
    df['all_three'].sum(),
]
colors = ['#bdc3c7', '#bdc3c7', '#bdc3c7', '#e67e22', '#c0392b']
bars = ax.bar(agree_labels, agree_counts, color=colors, alpha=0.85)
for bar, count in zip(bars, agree_counts):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 5,
            f'{count:,}', ha='center', fontsize=10, fontweight='bold')
ax.set_ylabel('Number of properties')
ax.set_title('Method Agreement  --  Anomaly Detection Cross-Validation', fontweight='bold', fontsize=13)
plt.tight_layout()
plt.show()

### Method agreement observations

The pairwise Jaccard overlap between IQR and Isolation Forest provides a measure of agreement between a simple statistical method and a machine learning approach. Properties flagged by all three methods represent the **highest-confidence anomalies** -- buildings where statistical, kernel-based, and tree-based approaches all agree that energy performance is abnormal.

This multi-method agreement acts as a substitute for labelled ground truth. In a real deployment, the high-confidence tier (all three methods agree) would be prioritised for survey and retrofit investment, while medium-confidence cases (two methods) would form a secondary shortlist.

## 8. Anomaly Profiling & Retrofit Prioritisation
What makes flagged buildings different? Which construction eras need the most intervention?

In [ ]:
# ── Feature comparison: normal vs anomalous ──────────────────────────────
# Using Isolation Forest labels as the primary method
normal = df[df['if_anomaly'] == 0]
anomalous = df[df['if_anomaly'] == 1]

comparison = pd.DataFrame({
    'Normal (median)': normal[epc_cols].median(),
    'Anomalous (median)': anomalous[epc_cols].median(),
    'Ratio': anomalous[epc_cols].median() / normal[epc_cols].median()
}).round(2)
comparison.index = [c.replace('_', ' ').title() for c in comparison.index]
print('=== Median Feature Values: Normal vs Anomalous Properties ===')
print(comparison.to_string())

# Bar chart: median comparison
fig, ax = plt.subplots(figsize=(12, 5))
x = np.arange(len(epc_cols))
width = 0.35
labels = [c.replace('_', ' ').title() for c in epc_cols]

# Normalise to percentage of normal median for visual comparison
normal_meds = normal[epc_cols].median().values
anom_meds = anomalous[epc_cols].median().values
ratios = anom_meds / normal_meds

ax.bar(x, [1]*len(epc_cols), width, label='Normal (baseline)', color='steelblue', alpha=0.7)
ax.bar(x + width, ratios, width, label='Anomalous (multiple of normal)',
       color='#c0392b', alpha=0.8)
ax.set_xticks(x + width/2)
ax.set_xticklabels(labels, rotation=30, ha='right', fontsize=8)
ax.set_ylabel('Multiple of normal median')
ax.axhline(1, color='grey', linestyle='--', linewidth=0.8)
ax.legend(fontsize=9)
ax.set_title('Anomalous vs Normal Properties  --  Feature Comparison (Isolation Forest)',
             fontweight='bold', fontsize=12)
plt.tight_layout()
plt.show()

### Anomaly profiling observations

Anomalous properties show median energy consumption, CO2 emissions, and heating costs that are multiples of the normal median. This confirms that the anomaly detection methods are identifying genuinely inefficient buildings, not statistical noise. The energy efficiency score (SAP) for anomalous properties is correspondingly lower, indicating worse overall building fabric and heating system performance.

In [ ]:
# ── Anomaly rate by construction era ──────────────────────────────────────
if 'construction_age_band' in df.columns:
    era_order = ['before 1900', '1900-1929', '1930-1949', '1950-1966',
                 '1967-1975', '1976-1982', '1983-1990', '1991-1995',
                 '1996-2002', '2003-2006', '2007-2011', '2012 onwards']
    era_present = [e for e in era_order if e in df['construction_age_band'].values]

    era_stats = df.groupby('construction_age_band').agg(
        total=('if_anomaly', 'count'),
        anomalous=('if_anomaly', 'sum')
    )
    era_stats['anomaly_rate'] = 100 * era_stats['anomalous'] / era_stats['total']
    era_stats = era_stats.reindex(era_present).dropna()

    fig, axes = plt.subplots(1, 2, figsize=(16, 6))

    # Left: anomaly rate by era
    colors = ['#c0392b' if r > era_stats['anomaly_rate'].median() else 'steelblue'
              for r in era_stats['anomaly_rate'].values]
    axes[0].barh(era_stats.index, era_stats['anomaly_rate'], color=colors, alpha=0.85)
    axes[0].set_xlabel('Anomaly rate (%)')
    axes[0].set_title('Anomaly Rate by Construction Era\n(red = above median rate)',
                      fontweight='bold')
    axes[0].invert_yaxis()
    axes[0].axvline(era_stats['anomaly_rate'].median(), color='grey',
                    linestyle='--', linewidth=1, alpha=0.7)

    # Right: anomaly count by era (absolute priority volume)
    axes[1].barh(era_stats.index, era_stats['anomalous'], color='#e67e22', alpha=0.85)
    axes[1].set_xlabel('Number of anomalous properties')
    axes[1].set_title('Retrofit Priority Volume by Construction Era', fontweight='bold')
    axes[1].invert_yaxis()

    plt.suptitle('Retrofit Prioritisation  --  Where to Invest First',
                 fontsize=14, fontweight='bold', y=1.02)
    plt.tight_layout()
    plt.savefig('nhs_retrofit_priority.png', dpi=150, bbox_inches='tight')
    plt.show()

    print('\n=== Anomaly rate by construction era ===')
    print(era_stats[['total', 'anomalous', 'anomaly_rate']].to_string())

### Retrofit prioritisation observations

Pre-1930 properties have the highest anomaly rate, consistent with solid-wall construction, poor insulation, and ageing heating systems. The rate falls progressively for newer construction eras, with properties built after 2006 showing the lowest anomaly rates -- reflecting modern Building Regulations (Part L) requirements.

For retrofit investment decisions, the **left chart** (rate) identifies which eras are most problematic proportionally, while the **right chart** (volume) shows where the absolute number of intervention candidates is highest. A local authority would use both: the rate chart to set eligibility criteria, and the volume chart to estimate programme scale and budget.

In [ ]:
# ── Anomaly rate by property type ──────────────────────────────────────────
if 'property_type' in df.columns:
    pt_stats = df.groupby('property_type').agg(
        total=('if_anomaly', 'count'),
        anomalous=('if_anomaly', 'sum')
    )
    pt_stats['anomaly_rate'] = 100 * pt_stats['anomalous'] / pt_stats['total']
    pt_stats = pt_stats.sort_values('anomaly_rate', ascending=False)

    fig, ax = plt.subplots(figsize=(10, 4))
    ax.barh(pt_stats.index, pt_stats['anomaly_rate'], color='steelblue', alpha=0.85)
    ax.set_xlabel('Anomaly rate (%)')
    ax.set_title('Anomaly Rate by Property Type', fontweight='bold')
    ax.invert_yaxis()
    plt.tight_layout()
    plt.show()

    print(pt_stats[['total', 'anomalous', 'anomaly_rate']].round(1).to_string())

In [ ]:
# ── Energy consumption distribution: normal vs anomalous ──────────────────
fig, ax = plt.subplots(figsize=(12, 5))
bins = np.linspace(0, df['energy_consumption_current'].quantile(0.99), 80)

ax.hist(normal['energy_consumption_current'], bins=bins, alpha=0.7,
        color='steelblue', label='Normal', density=True)
ax.hist(anomalous['energy_consumption_current'], bins=bins, alpha=0.7,
        color='#c0392b', label='Anomalous (Isolation Forest)', density=True)

ax.set_xlabel('Energy consumption (kWh/m2/yr)')
ax.set_ylabel('Density')
ax.set_title('Energy Consumption Distribution  --  Normal vs Anomalous Properties',
             fontweight='bold', fontsize=12)
ax.legend(fontsize=10)
plt.tight_layout()
plt.savefig('nhs_anomaly_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

## 9. Comparison of Methods and Final Recommendation

Three anomaly detection approaches were applied to real domestic EPC data from four English local authorities:

**IQR method** is simple and highly interpretable, allowing anomalies to be identified based on extreme values in individual features. It is useful for initial screening and straightforward to communicate to non-technical stakeholders such as local authority housing teams. However, it relies on manually selecting the simultaneous-flag threshold and cannot capture non-linear relationships between features.

**One-Class SVM** models a complex, non-linear decision boundary and captures multivariate patterns that the IQR method misses. However, it requires careful tuning of `nu` and `gamma` and is less intuitive to explain.

**Isolation Forest** also identified approximately 2% of properties as anomalous and was the easiest of the three to tune. The contamination parameter directly specifies the expected anomaly proportion, reducing iterative adjustment. Its results showed strong agreement with the IQR method, providing cross-validation confidence.

### Validation without labels

Since no ground-truth "bad building" labels exist, three validation strategies were used:
1. **IQR as statistical baseline** -- flags extreme values independently, providing an auditable reference.
2. **Method agreement as confidence signal** -- properties flagged by multiple methods represent the highest-confidence anomalies.
3. **PCA as sanity check** -- anomalous properties cluster in the high-consumption region of PC1, confirming the methods detect a real structural pattern.

### Anomaly profiling confirms real-world validity

Anomalous properties show median energy consumption and CO2 emissions that are multiples of the normal median. They are disproportionately concentrated in pre-1930 construction eras with solid-wall construction -- consistent with domain knowledge about the worst-performing segments of the English housing stock.

### Recommendation

**Isolation Forest is recommended as the primary anomaly detection method**, supported by IQR for interpretability when reporting to non-technical stakeholders. For operational deployment:

| Priority tier | Criterion | Action |
|---|---|---|
| **High confidence** | Flagged by all 3 methods | Immediate survey & retrofit assessment |
| **Medium confidence** | Flagged by any 2 methods | Secondary shortlist for outreach |
| **Watch list** | Flagged by 1 method only | Monitor; may warrant assessment if budget allows |

This tiered approach concentrates limited local authority survey budget on the buildings where intervention will have the most measurable impact on carbon emissions and fuel poverty.